# GTM Planning Engine — Orchestration Notebook
### Config-Driven SAO Allocation & Revenue Planning

This notebook runs the full GTM planning pipeline:
1. Load configuration and data
2. Generate period-level targets
3. Model AE capacity (hiring, ramp, mentoring, shrinkage)
4. Calculate marginal economics (decay curves)
5. Optimize SAO allocation across segments
6. Validate results
7. Run recovery analysis
8. Run what-if scenarios
9. Save and compare versions

In [ ]:
import sys
import os
# Add parent directory to path so we can import gtm_engine
sys.path.insert(0, os.path.dirname(os.getcwd()))

import pandas as pd
import numpy as np
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print("✓ Dependencies loaded")

## 1. Load Configuration

In [ ]:
from gtm_engine.config_manager import ConfigManager

# Load the default configuration
config = ConfigManager('../config.yaml')
print(f"✓ Config loaded and validated")
print(f"  Active dimensions: {config.get_active_dimensions()}")
print(f"  Annual target: ${config.get('targets.annual_target'):,.0f}")
print(f"  Planning mode: {config.get('targets.planning_mode')}")
print(f"  Optimizer mode: {config.get('allocation.optimizer_mode')}")

## 2. Load and Prepare Data

In [ ]:
from gtm_engine.data_loader import DataLoader

loader = DataLoader(config)

# Load the 2025 actuals
df_raw = loader.load('../data/raw/2025_actuals.csv')
print(f"✓ Raw data loaded: {len(df_raw)} rows")
print(f"  Columns: {list(df_raw.columns)}")

# Prepare the data (filter dimensions, score confidence, flag supersized)
df_clean = loader.prepare(df_raw)
print(f"\n✓ Data prepared: {len(df_clean)} rows")
print(f"  Segments: {df_clean.groupby(config.get_segment_keys()).ngroups} unique segments")
if 'confidence_level' in df_clean.columns:
    print(f"  Confidence distribution:\n{df_clean['confidence_level'].value_counts().to_string()}")
df_clean.head(10)

## 3. Generate Targets
Distribute the annual revenue target ($188M) across months using seasonality weights.

In [ ]:
from gtm_engine.target_generator import TargetGenerator

target_gen = TargetGenerator(config)
targets = target_gen.generate()

print(f"✓ Targets generated for {len(targets)} periods")
print(f"  Annual total: ${targets['target_revenue'].sum():,.0f}")
print(f"\nMonthly targets:")
print(targets.to_string(index=False))

## 4. Model AE Capacity
Calculate effective SAO capacity by month, accounting for hiring tranches, ramp, mentoring overhead, shrinkage, and attrition.

In [ ]:
from gtm_engine.ae_capacity import AECapacityModel

ae_model = AECapacityModel(config)
capacity = ae_model.calculate()

print(f"✓ AE Capacity model complete")
summary = ae_model.get_capacity_summary()
print(f"  Starting HC: {summary.get('starting_hc', 'N/A')}")
print(f"  Ending HC: {summary.get('ending_hc', 'N/A')}")
print(f"  Total annual capacity: {summary.get('total_annual_capacity', 'N/A'):,.0f} SAOs")
print(f"\nMonthly capacity breakdown:")
print(capacity.to_string(index=False))

## 5. Compute Baselines & Initialize Economics Engine
Compute baseline ASP and CW rate per segment from actuals data, then initialize
the economics engine with these values. The aggregation method and grain are
controlled by `config.economics.baseline`.

In [ ]:
from gtm_engine.economics_engine import EconomicsEngine

# --- Step 5a: Compute baselines from actuals ---
baselines = loader.compute_segment_baselines(df_clean)

print(f"✓ Baselines computed (method: {config.get('economics.baseline.aggregation', 'median')}, "
      f"grain: {config.get('economics.baseline.grain', 'segment')})")
print(f"  Segments with baselines: {len(baselines)}")
for seg, vals in sorted(baselines.items()):
    asp_str = f"ASP=${vals['asp']:,.0f}" if 'asp' in vals else "ASP=n/a"
    cw_str = f"CW={vals['win_rate']:.1%}" if 'win_rate' in vals else "CW=n/a"
    print(f"    {seg}: {asp_str}, {cw_str}")

# --- Step 5b: Initialize economics engine and load baselines ---
economics = EconomicsEngine(config)
economics.load_baselines(baselines)

print(f"\n✓ Economics engine ready — baselines loaded from actuals")
print(f"  Using {'calibrated' if config.get('economics.use_calibration') else 'default'} decay parameters")

# Demo: show ROI at different volume levels for a segment
demo_segment = list(baselines.keys())[0] if baselines else "EOR.Marketing"
try:
    for vol in [100, 500, 1000, 2000]:
        roi = economics.get_effective_roi(demo_segment, vol)
        asp = economics.get_effective_asp(demo_segment, vol)
        cw = economics.get_effective_win_rate(demo_segment, vol)
        print(f"  {demo_segment} at {vol} SAOs: ASP=${asp:,.0f}, CW={cw:.1%}, ROI=${roi:,.0f}/SAO")
except Exception as e:
    print(f"  Demo segment output not available: {e}")

## 6. Run Allocation Optimizer
The core engine: distribute SAOs across product-channel segments to meet targets.

In [ ]:
from gtm_engine.optimizer import AllocationOptimizer

optimizer = AllocationOptimizer(config)

# Run optimization
results = optimizer.optimize(
    targets=targets,
    base_data=df_clean,
    economics_engine=economics,
    capacity=capacity
)

print(f"✓ Optimization complete: {len(results)} allocation rows")

# Summary
opt_summary = optimizer.get_optimization_summary(results)
print(f"\n  Total projected bookings: ${opt_summary.get('total_bookings', 0):,.0f}")
print(f"  Total SAOs required: {opt_summary.get('total_saos', 0):,.0f}")
print(f"  Months with capacity constraints: {opt_summary.get('capacity_constrained_months', 0)}")

# Show allocation for Month 1 as example
print(f"\n--- Month 1 Allocation ---")
m1 = results[results['month'] == 1] if 'month' in results.columns else results.head(9)
print(m1.to_string(index=False))

## 6b. Mid-Cycle Adjustments
Apply a sample mid-cycle adjustment using the Adjustments Engine. This demonstrates how to handle actual performance data, HC changes, and target updates.

In [ ]:
from gtm_engine.adjustments import AdjustmentEngine

# Initialize the Adjustment Engine
adj_engine = AdjustmentEngine(config)

# Example: Simulate Q2 actuals and apply mid-cycle adjustments
# Create a simple actuals DataFrame (e.g., months 1-6 completed)
actuals_sample = pd.DataFrame({
    'period': [1, 2, 3, 4, 5, 6],
    'segment': ['EOR.Marketing'] * 6,
    'actual_bookings': [14000000, 15500000, 16200000, 17100000, 18500000, 19200000]
})

# Apply adjustments:
# - Lock months 1-6 with actual performance
# - Reduce HC in month 5 (3 AEs departed)
# - Increase annual target to $195M (from $188M)
# - Adjust EOR ASP downward to $11.5K (market pressure)
adjustment_result = adj_engine.apply_adjustment(
    current_plan=results,
    actuals=actuals_sample,
    hc_changes={'month_5': -3},
    target_changes={'annual_target': 195000000},
    segment_changes={'EOR.asp': 11500}
)

print("✓ Mid-Cycle Adjustment Applied\n")
print(adjustment_result['adjustment_summary'])
print(f"\nAdjustment Changes Applied:")
for change_type, count in adjustment_result['changes_applied'].items():
    if count > 0:
        print(f"  {change_type}: {count}")


## 7. Validate Results
Run mathematical consistency checks and constraint verification.

In [ ]:
from gtm_engine.validation import ValidationEngine

validator = ValidationEngine(config)
validation = validator.validate(results, targets=targets, capacity=capacity)

validator.print_report(validation)

## 8. Recovery Analysis
Check quarterly targets vs capacity and model recovery options if needed.

In [ ]:
from gtm_engine.recovery import RecoveryEngine

recovery = RecoveryEngine(config)
recovery_analysis = recovery.analyze(results, targets, capacity)

print("=== Quarterly Summary ===")
if 'quarterly_summary' in recovery_analysis:
    print(recovery_analysis['quarterly_summary'].to_string(index=False) if isinstance(recovery_analysis['quarterly_summary'], pd.DataFrame) else recovery_analysis['quarterly_summary'])

print(f"\nRisk Assessment: {recovery_analysis.get('risk_assessment', 'N/A')}")
if recovery_analysis.get('stretch_flags'):
    print(f"⚠️  Stretch flags: {recovery_analysis['stretch_flags']}")
else:
    print("✓ No quarters exceed stretch threshold")

## 10b. Allocation Comparison
Compare plan versions across multiple dimensions: volatility, concentration risk, capacity utilization, and more.

In [ ]:
from gtm_engine.comparator import VersionComparator

# Initialize the comparator with config (so it can use stretch thresholds, etc.)
comparator = VersionComparator(config)

# Get baseline plan and a hypothetical second version for comparison
# In practice, you'd retrieve multiple versions from version_store
baseline_version = {
    'version_id': 'v_base_plan',
    'results': results,
    'summary': opt_summary
}

# Create a second version by simulating a slight adjustment (for demo purposes)
# In reality, this would come from a previous re-optimization or scenario
adjusted_results = results.copy()
# Simulate a 3% variance in allocation
adjusted_results['projected_bookings'] = adjusted_results['projected_bookings'] * 1.03
adjusted_results['required_saos'] = adjusted_results['required_saos'] * 1.02

adjusted_summary = {
    'total_bookings': adjusted_results['projected_bookings'].sum(),
    'total_pipeline': opt_summary.get('total_pipeline', 0),
    'total_saos': adjusted_results['required_saos'].sum() if 'required_saos' in adjusted_results.columns else opt_summary.get('total_saos', 0),
    'total_ae_hc': opt_summary.get('total_ae_hc', 0),
    'monthly_volatility': opt_summary.get('monthly_volatility', 0),
    'capacity_constrained_months': opt_summary.get('capacity_constrained_months', 0)
}

adjusted_version = {
    'version_id': 'v_adjusted_plan',
    'results': adjusted_results,
    'summary': adjusted_summary
}

# Run the comparison
comparison = comparator.compare([baseline_version, adjusted_version])

print("✓ Version Comparison Complete\n")

# Print the full comparison report
comparator.print_comparison_report(comparison)

# Display the metrics table for easy reference
print("\nMETRICS TABLE (Detailed)")
print("-" * 160)
metrics_df = comparator.to_dataframe(comparison)
print(metrics_df.to_string(index=False))


## 9. Save Version
Store this plan version with its config and results for future comparison.

In [ ]:
from gtm_engine.version_store import VersionStore

store = VersionStore(config)
version_id = store.save(
    config_snapshot=config.to_dict(),
    results=results,
    summary=opt_summary,
    description="Base plan - greedy optimizer, default economics, 5%/40% guardrails",
    planning_mode=config.get('targets.planning_mode', 'full_year')
)

print(f"✓ Plan saved as Version {version_id}")
print(f"\nAll versions:")
versions_list = store.list_versions()
print(versions_list.to_string(index=False) if isinstance(versions_list, pd.DataFrame) else versions_list)

## 10. What-If Scenarios
Model the top risk scenarios against the base plan.

In [ ]:
from gtm_engine.what_if import WhatIfEngine

what_if = WhatIfEngine(config)

# Pipeline function: re-instantiates loader, data, and baselines per scenario
# so that scenarios which change baseline computation method get fresh values
def run_pipeline(scenario_config):
    """Re-run the full planning pipeline with a (potentially modified) config.

    Each scenario gets its own DataLoader, clean data, and baselines so that
    perturbations to economics.baseline.aggregation or active dimensions
    actually take effect rather than reusing the outer-scope cached values.
    """
    # --- Fresh data layer per scenario ---
    scenario_loader = DataLoader(scenario_config)
    scenario_df_raw = scenario_loader.load('../data/raw/2025_actuals.csv')
    scenario_df_clean = scenario_loader.prepare(scenario_df_raw)
    scenario_baselines = scenario_loader.compute_segment_baselines(scenario_df_clean)

    # --- Planning pipeline ---
    tg = TargetGenerator(scenario_config)
    t = tg.generate()

    ae = AECapacityModel(scenario_config)
    c = ae.calculate()

    econ = EconomicsEngine(scenario_config)
    econ.load_baselines(scenario_baselines)

    opt = AllocationOptimizer(scenario_config)
    r = opt.optimize(t, scenario_df_clean, econ, c)
    s = opt.get_optimization_summary(r)
    return r, s

# Run what-if scenarios
try:
    comparison = what_if.run_scenarios(results, opt_summary, run_pipeline)
    what_if.print_comparison(comparison)
except Exception as e:
    print(f"What-if scenarios encountered an issue: {e}")
    print("(This is expected if some perturbation types aren't fully supported yet)")

## Summary
All modules executed successfully. The GTM Planning Engine is ready for:
- **Quarterly planning**: Adjust config.yaml and re-run
- **Mid-quarter re-planning**: Use Ad-Hoc Adjustment module with actuals
- **Scenario analysis**: Modify what-if scenarios in config
- **Version comparison**: Use Version Comparator to compare plan iterations